# CardioIA — Ir Além 1: extração clínica com IA generativa

Pipeline que transforma narrativas livres (prontuário **sintético**) em JSON estrito, validado por **Pydantic v2**.

**Aviso ético:** este assistente não substitui atendimento médico. Em emergências, ligue 192 (SAMU).

Sem `OPENAI_API_KEY` o extrator heurístico cobre todos os casos de `dataset_casos_teste.json`.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if (ROOT / "clinical_extraction.py").exists():
    sys.path.insert(0, str(ROOT))
else:
    sys.path.insert(0, str(ROOT / "ir_alem_1_genai"))

from clinical_extraction import extract_clinical_record, run_dataset, DISCLAIMER

print(DISCLAIMER)
dataset = Path("dataset_casos_teste.json")
if not dataset.exists():
    dataset = ROOT / "ir_alem_1_genai" / "dataset_casos_teste.json"
print("Dataset:", dataset.resolve())

## Casos sintéticos e extração em lote

In [ ]:
resultados = run_dataset(dataset)
for row in resultados:
    print(f"{row['id_caso']:22} esperado={row['risco_esperado']:12} obtido={row['classificacao_risco']:12} ok={row['bate_esperado']}")
    print("  QP:", row["queixa_principal"][:90])
    sv = row["sinais_vitais"]
    print(f"  PA {sv['pressao_sistolica']:.0f}/{sv['pressao_diastolica']:.0f}  FC {sv['frequencia_cardiaca']:.0f}  SpO2 {sv.get('spo2')}")
print("\nAcurácia de risco no dataset:", sum(r["bate_esperado"] for r in resultados), "/", len(resultados))

## Extração pontual (exemplo de emergência)

In [ ]:
texto = Path(dataset).read_text(encoding="utf-8")
casos = json.loads(texto)
emergencia = next(c for c in casos if c["id"] == "CASO-EMERGENCIA-01")
modelo = extract_clinical_record(emergencia["narrativa"], prefer_llm=False)
print(modelo.model_dump_json(indent=2))